In [0]:
from LDCDataAccessLayerPy import KeyVaultManager, SharePointManager, SqlManager, databricks_init
from datetime import datetime, timedelta
from LDCDataAccessLayerPy import databricks_init, DataLakeManagerGen2
from io import BytesIO
import LDCDataAccessLayerPy
#Initiate the secret to access KeyVault secrets
databricks_init(dbutils, 'GO')
sp_mgr = SharePointManager()
sql_mgr = SqlManager()

import logging
logger = spark._jvm.org.apache.log4j
logging.getLogger("py4j").setLevel(logging.ERROR)

import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.colors import ListedColormap
from sklearn.cluster import KMeans

from datetime import datetime, timedelta
import re

url = "https://ldcom365.sharepoint.com"

In [0]:

df=sp_mgr.read_pd_from_excel('/sites/GRP-TradingLineups/Lineups/merged/history and lineups.xlsx')

In [0]:
df['Date'] = pd.to_datetime(df['Date'])

# Filter for the selected products
selected_products = ['SOYBEANS', 'SBM', 'Corn', 'Wheat', 'Barley', 'SBO']
df = df[df['Product'].isin(selected_products)]

# Sort DataFrame
df = df.sort_values(by=['Year', 'Month', 'Product', 'Status', 'Date'])

# Separate cumulative sums
df_sailed = (
    df[df['Status'] == 'SAILED']
    .groupby(['Year', 'Month', 'Product', df['Date'].dt.day])['Quantity']
    .sum()
    .groupby(level=[0, 1, 2])
    .cumsum()
    .unstack(level=2, fill_value=0)
)

df_all_status = (
    df.groupby(['Year', 'Month', 'Product', df['Date'].dt.day])['Quantity']
    .sum()
    .groupby(level=[0, 1, 2])
    .cumsum()
    .unstack(level=2, fill_value=0)
)

# Combine into a single DataFrame with MultiIndex columns
df_combined = pd.concat(
    {
        'SAILED': df_sailed,
        'ALL_STATUS': df_all_status,
    },
    axis=1
)

# Flatten the columns for better readability
df_combined.columns = [
    f"{col[1]} {col[0]}" for col in df_combined.columns.to_flat_index()
]

# Reset the index to include Year, Month, and Day as columns
df_combined = df_combined.reset_index()
df_combined = df_combined.rename(columns={'Date': 'Day'})
# Create a Date column from Year, Month, and Day

df_combined['Date'] = pd.to_datetime(df_combined[['Year', 'Month', 'Day']],errors='coerce')
# Drop rows where 'Date' is NaT (invalid date conversions)
df_combined = df_combined[df_combined['Date'].notna()]
#sp_mgr.save_pd_to_excel('/sites/GRP-TradingLineups/Lineups/merged/cumsum.xlsx',df_combined,index=False)

In [0]:
melted = pd.melt(df_combined, id_vars=['Year', 'Month', 'Day', 'Date'], 
                 value_vars=['Barley SAILED', 'Corn SAILED', 'SBM SAILED', 'SBO SAILED', 'SOYBEANS SAILED', 'Wheat SAILED', 
                             'Barley ALL_STATUS', 'Corn ALL_STATUS', 'SBM ALL_STATUS', 'SBO ALL_STATUS', 'SOYBEANS ALL_STATUS', 'Wheat ALL_STATUS'], 
                 var_name='Product_Status', value_name='CumSum')

# Extracting the product and status
melted['Product'] = melted['Product_Status'].str.extract(r'([A-Za-z]+)')
melted['Status'] = melted['Product_Status'].str.extract(r'(SAILED|ALL_STATUS)')

# Dropping the Product_Status column
melted.drop(columns=['Product_Status'], inplace=True)

# Reordering columns
melted = melted[['Year', 'Month', 'Day', 'Date', 'Product', 'CumSum', 'Status']]
melted['CumSum'] = melted.groupby(['Year', 'Month', 'Product', 'Status'])['CumSum'].apply(lambda group: group.replace(0, method='ffill'))



In [0]:
def pivot_dataframe(df):
    # Identify columns for pivoting
    products = ['Barley', 'Corn', 'SBM', 'SBO', 'SOYBEANS', 'Wheat']
    status_types = ['SAILED', 'ALL_STATUS']

    # Melt the DataFrame to a long format
    melted = pd.melt(
        df,
        id_vars=['Year', 'Month', 'Day', 'Date'],
        value_vars=[f"{product} {status}" for product in products for status in status_types],
        var_name='Product_Status',
        value_name='CumSum'
    )

    # Split the "Product Status" column into "Product" and "Status"
    melted[['Product', 'Status']] = melted['Product_Status'].str.split(' ', n=1, expand=True)

    # Normalize "ALL_STATUS" to "ALL"
    melted['Status'] = melted['Status'].replace({'ALL_STATUS': 'ALL'})

    # Drop the temporary "Product_Status" column
    melted = melted.drop(columns=['Product_Status'])

    # Return the transformed DataFrame
    return melted

In [0]:
df_combined=pivot_dataframe(df_combined)

df_combined["CumSum"] = pd.to_numeric(df_combined["CumSum"], errors="coerce")

df_combined.loc[(df_combined["Day"] != 1) &  # Avoid modifying Day 1 values
    (df_combined["CumSum"] == 0) &  # Target rows where CumSum is 0
    (df_combined["Month"] == df_combined["Month"].shift(1))&(df_combined["Product"] == df_combined["Product"].shift(1))&(df_combined["Status"] == df_combined["Status"].shift(1)),  # Ensure same month as the previous row
    "CumSum"
] = df_combined["CumSum"].shift(1)  # Assign the previous row's CumSum

for i in range(1, len(df_combined)):
    if (df_combined.loc[i, "Year"] == df_combined.loc[i-1, "Year"] and
        df_combined.loc[i, "Month"] == df_combined.loc[i-1, "Month"] and
        df_combined.loc[i, "Product"] == df_combined.loc[i-1, "Product"] and
        df_combined.loc[i, "Status"] == df_combined.loc[i-1, "Status"] and
        df_combined.loc[i, "CumSum"] == 0):
        
        # Keep the previous non-zero quantity
        df_combined.loc[i, "CumSum"] = df_combined.loc[i-1, "CumSum"]

In [0]:
sp_mgr.save_pd_to_excel('/sites/GRP-TradingLineups/Lineups/merged/cumsum.xlsx',df_combined,index=False)